# 01 — Tabular data preprocessing

## From raw data to a reproducible processed table

This notebook is the first practical example in `notebooks/tabular/task1_alzheimer/`. It implements the workflow described in `docs/tabular_data_preprocessing.md` using the synthetic Alzheimer dataset included in this repository.

The goal is deliberately narrow: **read, inspect, validate, clean, and export a tabular dataset before any exploratory data analysis or machine-learning step**.

By the end of the notebook, we will have:

1. loaded the raw table and its metadata;
2. verified the expected schema and column roles;
3. standardized column names and data types;
4. checked missing values, duplicated records, and duplicated identifiers;
5. validated categorical encodings and simple numerical constraints;
6. checked the internal consistency of derived variables;
7. exported a processed table without modifying the raw source;
8. reloaded the exported file to verify that it can be reproduced correctly.

> **Important:** this is a synthetic educational dataset. The checks used here are data-quality checks, not clinical criteria, and the notebook does not perform medical interpretation.


## 1. Imports

Only a small set of libraries is required for this stage. `pandas` is used for tabular data, `numpy` for numerical validation, `json` for the metadata file, and `pathlib` for portable file paths.

Keeping the dependency set small makes the preprocessing workflow easier to reproduce and debug.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)


## 2. Locate the repository and define paths

Hard-coded absolute paths such as `/home/user/...` make notebooks difficult to share. Instead, we locate the repository by searching upward from the current working directory and then construct all paths relative to the repository root.

This allows the notebook to work whether Jupyter was launched from the repository root or from the notebook directory.


In [2]:
def find_project_root(start: Path) -> Path:
    # Return the nearest parent directory containing the expected project folders.
    start = start.resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository root. Run the notebook from inside the project."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

RAW_DIR = PROJECT_ROOT / "data" / "synthetic_alzheimer" / "raw_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "synthetic_alzheimer" / "processed_data"

RAW_DATA_PATH = RAW_DIR / "synthetic_alzheimer_patients_1000.csv"
METADATA_PATH = RAW_DIR / "metadata.json"

PROCESSED_DATA_PATH = PROCESSED_DIR / "synthetic_alzheimer_patients_1000_processed.csv"
REPORT_PATH = PROCESSED_DIR / "preprocessing_report.json"

print("Repository root:", PROJECT_ROOT.name)
print("Raw data:", RAW_DATA_PATH.relative_to(PROJECT_ROOT))
print("Metadata:", METADATA_PATH.relative_to(PROJECT_ROOT))
print("Processed output:", PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT))


Repository root: demos_machine_learning_2026
Raw data: data/synthetic_alzheimer/raw_data/synthetic_alzheimer_patients_1000.csv
Metadata: data/synthetic_alzheimer/raw_data/metadata.json
Processed output: data/synthetic_alzheimer/processed_data/synthetic_alzheimer_patients_1000_processed.csv


## 3. Read the metadata before modifying the table

The metadata defines the intended role of the columns. This is preferable to guessing a variable's role from its name or its numerical representation.

In this example, the metadata identifies:

- the target column;
- the identifier column;
- categorical columns;
- numerical columns.

These roles will be used throughout the preprocessing checks.


In [3]:
with METADATA_PATH.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

required_metadata_keys = {
    "label",
    "col_id",
    "categorical_column",
    "numerical_column",
}

missing_metadata_keys = required_metadata_keys - set(metadata)
if missing_metadata_keys:
    raise KeyError(f"Missing metadata keys: {sorted(missing_metadata_keys)}")

TARGET_COLUMN = metadata["label"]
ID_COLUMN = metadata["col_id"]
CATEGORICAL_COLUMNS = list(metadata["categorical_column"])
NUMERICAL_COLUMNS = list(metadata["numerical_column"])

print("Identifier column:", ID_COLUMN)
print("Target column:", TARGET_COLUMN)
print("Number of categorical columns:", len(CATEGORICAL_COLUMNS))
print("Number of numerical columns:", len(NUMERICAL_COLUMNS))


Identifier column: patient_id
Target column: alzheimer
Number of categorical columns: 19
Number of numerical columns: 10


## 4. Read the raw dataset

Missing information may be represented using different text tokens in real datasets. We define a small set of common tokens at import time so that they are interpreted consistently as missing values.

The raw file is **read only**. It will never be overwritten by this notebook.


In [4]:
MISSING_TOKENS = [
    "NA",
    "N/A",
    "null",
    "None",
    "missing",
    "unknown",
]

raw_df = pd.read_csv(
    RAW_DATA_PATH,
    na_values=MISSING_TOKENS,
    keep_default_na=True,
)

print(f"Loaded {raw_df.shape[0]} rows and {raw_df.shape[1]} columns.")
display(raw_df.head())


Loaded 1000 rows and 31 columns.


,patient_id,age,sex_female,height_cm,weight_kg,bmi,waist_cm,hip_cm,waist_hip_ratio,systolic_bp_mmHg,diastolic_bp_mmHg,heart_rate_bpm,smoker,alcohol_risk,physical_inactivity,hypertension,diabetes_type2,dyslipidemia,cardiovascular_disease,stroke_history,depression_history,traumatic_brain_injury,family_history_alzheimer,sleep_disorder,chronic_kidney_disease,hearing_loss,asthma,allergy_medication,allergy_food,allergy_environmental,alzheimer
0,P0001,75,1,158.7,69.5,27.6,116.6,136.3,0.855,137,77,81,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
1,P0002,50,1,156.5,70.3,28.7,122.1,130.5,0.936,99,78,62,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,P0003,82,0,169.4,76.6,26.7,117.8,118.3,0.996,153,85,73,1,0,1,1,0,0,1,1,0,0,1,1,0,0,0,0,1,0,1
3,P0004,72,1,165.2,80.8,29.6,110.1,123.4,0.892,121,75,73,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1
4,P0005,64,1,151.0,75.4,33.0,127.2,144.4,0.881,153,74,84,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0


## 5. Verify the schema

A file can be read successfully and still have the wrong structure. We therefore compare the columns found in the CSV against the columns declared in the metadata.

This is a **schema check**, not an analysis of the dataset. It answers whether the expected variables are present and whether unexpected variables appeared in the input file.


In [5]:
expected_columns_raw = {
    ID_COLUMN,
    TARGET_COLUMN,
    *CATEGORICAL_COLUMNS,
    *NUMERICAL_COLUMNS,
}
observed_columns = set(raw_df.columns)

missing_columns = sorted(expected_columns_raw - observed_columns)
unexpected_columns = sorted(observed_columns - expected_columns_raw)

schema_summary = pd.DataFrame(
    {
        "check": ["Missing expected columns", "Unexpected columns"],
        "result": [missing_columns or "None", unexpected_columns or "None"],
    }
)

display(schema_summary)

if missing_columns or unexpected_columns:
    raise ValueError(
        "The input schema does not match metadata. "
        f"Missing={missing_columns}, unexpected={unexpected_columns}"
    )


,check,result
0,Missing expected columns,None
1,Unexpected columns,None


## 6. Define and inspect column roles

Separating identifiers, numerical variables, categorical variables, and the target prevents accidental misuse later. In particular, an identifier should not become a predictive feature simply because it appears in the same table.

The table below is a structural description of the columns. It is not an exploratory statistical summary.


In [6]:
def column_role(column: str) -> str:
    if column == ID_COLUMN:
        return "identifier"
    if column == TARGET_COLUMN:
        return "target"
    if column in CATEGORICAL_COLUMNS:
        return "categorical"
    if column in NUMERICAL_COLUMNS:
        return "numerical"
    return "unassigned"


role_table = pd.DataFrame(
    {
        "column": raw_df.columns,
        "role": [column_role(column) for column in raw_df.columns],
        "raw_dtype": [str(raw_df[column].dtype) for column in raw_df.columns],
    }
)

display(role_table)


,column,role,raw_dtype
0,patient_id,identifier,str
1,age,numerical,int64
2,sex_female,categorical,int64
3,height_cm,numerical,float64
4,weight_kg,numerical,float64
5,bmi,numerical,float64
6,waist_cm,numerical,float64
7,hip_cm,numerical,float64
8,waist_hip_ratio,numerical,float64
9,systolic_bp_mmHg,numerical,int64


## 7. Standardize column names

Column names are normalized to a simple `snake_case` convention by trimming whitespace, converting to lowercase, and replacing spaces or hyphens with underscores.

Before renaming, we check that normalization does not cause two different original names to collapse into the same name. Silent column-name collisions are dangerous because they can make variables ambiguous.


In [7]:
def normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


normalized_columns = [normalize_column_name(column) for column in raw_df.columns]

if len(normalized_columns) != len(set(normalized_columns)):
    raise ValueError("Column-name normalization would create duplicated column names.")

rename_map = dict(zip(raw_df.columns, normalized_columns))
changed_names = {
    old: new
    for old, new in rename_map.items()
    if old != new
}

print("Column names changed:", changed_names or "None")

df = raw_df.rename(columns=rename_map).copy()

# Keep metadata roles synchronized with the normalized schema.
ID_COLUMN = normalize_column_name(ID_COLUMN)
TARGET_COLUMN = normalize_column_name(TARGET_COLUMN)
CATEGORICAL_COLUMNS = [normalize_column_name(column) for column in CATEGORICAL_COLUMNS]
NUMERICAL_COLUMNS = [normalize_column_name(column) for column in NUMERICAL_COLUMNS]
FINAL_COLUMN_ORDER = normalized_columns
EXPECTED_COLUMNS = {
    ID_COLUMN,
    TARGET_COLUMN,
    *CATEGORICAL_COLUMNS,
    *NUMERICAL_COLUMNS,
}


Column names changed: {'systolic_bp_mmHg': 'systolic_bp_mmhg', 'diastolic_bp_mmHg': 'diastolic_bp_mmhg'}


## 8. Convert columns according to their declared roles

Data types should follow the meaning of each variable rather than whatever type pandas happened to infer.

We therefore:

- keep the identifier as text;
- convert declared numerical variables to numeric values;
- convert the categorical indicators and target to integer values.

Conversion failures are recorded before validation. A value that cannot be converted should be investigated rather than silently accepted.


In [8]:
conversion_failures = {}

# Identifier: preserve as text and remove accidental surrounding whitespace.
df[ID_COLUMN] = df[ID_COLUMN].astype("string").str.strip()

# Numerical variables.
for column in NUMERICAL_COLUMNS:
    original_non_missing = df[column].notna()
    converted = pd.to_numeric(df[column], errors="coerce")
    failed = original_non_missing & converted.isna()
    conversion_failures[column] = int(failed.sum())
    df[column] = converted

# Categorical indicators and the target are encoded as integers in this teaching dataset.
for column in [*CATEGORICAL_COLUMNS, TARGET_COLUMN]:
    original_non_missing = df[column].notna()
    converted = pd.to_numeric(df[column], errors="coerce")
    failed = original_non_missing & converted.isna()
    conversion_failures[column] = int(failed.sum())
    df[column] = converted.astype("Int64")

failure_table = pd.DataFrame(
    {
        "column": list(conversion_failures),
        "conversion_failures": list(conversion_failures.values()),
    }
)

failure_table = failure_table[failure_table["conversion_failures"] > 0]
display(failure_table if not failure_table.empty else pd.DataFrame({"conversion_failures": ["None"]}))

if any(conversion_failures.values()):
    raise ValueError("At least one value could not be converted to its expected data type.")


,conversion_failures
0,None


## 9. Check missing values

At this stage we identify missing information; we do **not** automatically replace it with zero, a mean, or another value.

Imputation can affect model evaluation and must be handled later within the appropriate train/test workflow when needed.


In [9]:
missing_summary = (
    df.isna()
    .sum()
    .rename("missing_values")
    .to_frame()
)

missing_summary = missing_summary[missing_summary["missing_values"] > 0]

display(
    missing_summary
    if not missing_summary.empty
    else pd.DataFrame({"missing_values": ["None"]})
)


,missing_values
0,None


## 10. Check exact duplicates and duplicated identifiers

Two different questions must be separated:

1. Are there rows that are exact copies of one another?
2. Is the identifier repeated across different rows?

For this dataset, `patient_id` is expected to identify one row uniquely. Exact duplicate rows can be removed reproducibly, but repeated identifiers with different content would require manual investigation rather than automatic deletion.


In [10]:
exact_duplicate_count = int(df.duplicated().sum())
duplicated_id_count = int(df[ID_COLUMN].duplicated().sum())

print("Exact duplicated rows:", exact_duplicate_count)
print("Duplicated identifiers:", duplicated_id_count)

if exact_duplicate_count > 0:
    df = df.drop_duplicates().copy()
    print(f"Removed {exact_duplicate_count} exact duplicated rows.")

# Re-check identifiers after removing exact duplicates.
duplicated_id_count_after_exact_dedup = int(df[ID_COLUMN].duplicated().sum())
if duplicated_id_count_after_exact_dedup > 0:
    duplicated_ids = df.loc[
        df[ID_COLUMN].duplicated(keep=False),
        [ID_COLUMN],
    ]
    display(duplicated_ids)
    raise ValueError(
        "Repeated patient identifiers remain after exact deduplication and require review."
    )


Exact duplicated rows: 0
Duplicated identifiers: 0


## 11. Validate categorical and target encodings

The categorical variables and target in this synthetic dataset are binary indicators encoded as `0` and `1`.

We verify the allowed domain explicitly. This prevents unexpected codes such as `2`, `-1`, or text labels from entering later stages unnoticed.


In [11]:
ALLOWED_BINARY_VALUES = {0, 1}
invalid_binary_values = {}

for column in [*CATEGORICAL_COLUMNS, TARGET_COLUMN]:
    observed = set(df[column].dropna().astype(int).unique().tolist())
    invalid = sorted(observed - ALLOWED_BINARY_VALUES)
    if invalid:
        invalid_binary_values[column] = invalid

print("Invalid binary encodings:", invalid_binary_values or "None")

if invalid_binary_values:
    raise ValueError(f"Invalid binary encodings found: {invalid_binary_values}")


Invalid binary encodings: None


## 12. Apply simple numerical sanity checks

Preprocessing should detect mechanically impossible values without turning into clinical interpretation.

For this teaching dataset we use only broad sanity constraints:

- age cannot be negative;
- body measurements, ratios, blood-pressure measurements, and heart rate must be positive;
- all numerical values must be finite.

These checks are intentionally conservative. They are **not diagnostic thresholds** and should not be interpreted as clinical rules.


In [12]:
invalid_numeric_checks = {}

# All declared numerical columns should contain finite values when they are not missing.
for column in NUMERICAL_COLUMNS:
    non_missing = df[column].dropna()
    non_finite_count = int((~np.isfinite(non_missing)).sum())
    if non_finite_count:
        invalid_numeric_checks[f"{column}: non-finite"] = non_finite_count

# Broad mechanical constraints for this dataset.
if int((df["age"].dropna() < 0).sum()):
    invalid_numeric_checks["age: negative"] = int((df["age"].dropna() < 0).sum())

positive_columns = [
    "height_cm",
    "weight_kg",
    "bmi",
    "waist_cm",
    "hip_cm",
    "waist_hip_ratio",
    "systolic_bp_mmhg",
    "diastolic_bp_mmhg",
    "heart_rate_bpm",
]

for column in positive_columns:
    invalid_count = int((df[column].dropna() <= 0).sum())
    if invalid_count:
        invalid_numeric_checks[f"{column}: non-positive"] = invalid_count

print("Invalid numerical values:", invalid_numeric_checks or "None")

if invalid_numeric_checks:
    raise ValueError(f"Numerical sanity checks failed: {invalid_numeric_checks}")


Invalid numerical values: None


## 13. Check internal consistency of derived variables

Some columns can be checked against other columns without performing statistical analysis.

Here:

- BMI can be recomputed from weight and height;
- waist-to-hip ratio can be recomputed from waist and hip circumference.

The stored values are rounded, so the comparison uses small numerical tolerances rather than requiring exact floating-point equality.


In [13]:
recomputed_bmi = df["weight_kg"] / (df["height_cm"] / 100.0) ** 2
recomputed_waist_hip_ratio = df["waist_cm"] / df["hip_cm"]

bmi_consistent = np.allclose(
    df["bmi"].to_numpy(dtype=float),
    recomputed_bmi.to_numpy(dtype=float),
    atol=0.1,
    rtol=0,
    equal_nan=True,
)

waist_hip_consistent = np.allclose(
    df["waist_hip_ratio"].to_numpy(dtype=float),
    recomputed_waist_hip_ratio.to_numpy(dtype=float),
    atol=0.002,
    rtol=0,
    equal_nan=True,
)

consistency_checks = pd.DataFrame(
    {
        "derived_variable": ["bmi", "waist_hip_ratio"],
        "consistent_with_source_columns": [bmi_consistent, waist_hip_consistent],
    }
)

display(consistency_checks)

if not bmi_consistent or not waist_hip_consistent:
    raise ValueError("At least one derived variable is inconsistent with its source columns.")


,derived_variable,consistent_with_source_columns
0,bmi,True
1,waist_hip_ratio,True


## 14. Final structural validation

Before exporting, we verify that the processed table still satisfies the intended contract.

The assertions below are useful because a notebook should fail loudly when an assumption is violated. A silent preprocessing error can propagate into every later analysis and model.


In [14]:
# Preserve the original column order.
df = df.loc[:, FINAL_COLUMN_ORDER].reset_index(drop=True)

assert list(df.columns) == FINAL_COLUMN_ORDER, "Column order changed unexpectedly."
assert set(df.columns) == EXPECTED_COLUMNS, "Final schema differs from metadata."
assert df[ID_COLUMN].notna().all(), "Missing identifiers remain."
assert df[ID_COLUMN].is_unique, "Identifiers are not unique."
assert df.duplicated().sum() == 0, "Exact duplicate rows remain."
assert not any(conversion_failures.values()), "Data-type conversion failures remain."
assert not invalid_binary_values, "Invalid categorical encodings remain."
assert not invalid_numeric_checks, "Invalid numerical values remain."
assert bmi_consistent and waist_hip_consistent, "Derived-variable checks failed."

final_validation = pd.DataFrame(
    {
        "property": [
            "Rows",
            "Columns",
            "Exact duplicate rows",
            "Duplicated identifiers",
            "Missing values",
        ],
        "value": [
            df.shape[0],
            df.shape[1],
            int(df.duplicated().sum()),
            int(df[ID_COLUMN].duplicated().sum()),
            int(df.isna().sum().sum()),
        ],
    }
)

display(final_validation)


,property,value
0,Rows,1000
1,Columns,31
2,Exact duplicate rows,0
3,Duplicated identifiers,0
4,Missing values,0


## 15. Export the processed dataset

The processed table is written to a separate directory. The raw file is preserved unchanged.

The notebook also writes a compact JSON report recording the input/output locations and the main preprocessing checks. This report is not an analysis of the dataset; it is an audit trail for reproducibility.


In [15]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DATA_PATH, index=False)

preprocessing_report = {
    "dataset": "synthetic_alzheimer",
    "input_file": str(RAW_DATA_PATH.relative_to(PROJECT_ROOT)),
    "metadata_file": str(METADATA_PATH.relative_to(PROJECT_ROOT)),
    "output_file": str(PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT)),
    "input_shape": [int(raw_df.shape[0]), int(raw_df.shape[1])],
    "output_shape": [int(df.shape[0]), int(df.shape[1])],
    "identifier_column": ID_COLUMN,
    "target_column": TARGET_COLUMN,
    "numerical_columns": NUMERICAL_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "exact_duplicate_rows_found": exact_duplicate_count,
    "duplicated_identifiers_found": duplicated_id_count,
    "missing_values_after_preprocessing": int(df.isna().sum().sum()),
    "bmi_consistency_check": bool(bmi_consistent),
    "waist_hip_ratio_consistency_check": bool(waist_hip_consistent),
}

with REPORT_PATH.open("w", encoding="utf-8") as file:
    json.dump(preprocessing_report, file, indent=2)

print("Processed dataset saved to:")
print(PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT))
print("Preprocessing report saved to:")
print(REPORT_PATH.relative_to(PROJECT_ROOT))


Processed dataset saved to:
data/synthetic_alzheimer/processed_data/synthetic_alzheimer_patients_1000_processed.csv
Preprocessing report saved to:
data/synthetic_alzheimer/processed_data/preprocessing_report.json


## 16. Reload the exported file

Writing a file successfully is not enough. We reload it and compare its structure and values with the in-memory processed table.

This final round-trip check confirms that the exported CSV can serve as the input to later notebooks.


In [16]:
reloaded_df = pd.read_csv(PROCESSED_DATA_PATH)

pd.testing.assert_frame_equal(
    df.reset_index(drop=True),
    reloaded_df.reset_index(drop=True),
    check_dtype=False,
)

print("Round-trip validation: PASS")
print("Reloaded shape:", reloaded_df.shape)


Round-trip validation: PASS
Reloaded shape: (1000, 31)


## 17. What was intentionally not done here?

This notebook stops after producing a structurally validated processed table.

It does **not** perform:

- distribution analysis;
- class-balance analysis;
- correlation analysis;
- outlier analysis based on statistical distributions;
- visualization;
- feature selection;
- imputation for modeling;
- scaling or normalization for modeling;
- train/test splitting;
- model training or evaluation.

Those operations answer different questions and belong to later stages of the workflow.

The important result of this notebook is therefore not a statistical conclusion. It is a **reproducible transition from the immutable raw dataset to a validated processed dataset** that can safely be used by the next exercise.
